# 02 - Detección de fraude financiero con Random Forest

## Caso fintech
Este notebook simula transacciones financieras y predice si una transacción es fraudulenta.

## Objetivo
Entrenar un modelo de clasificación binaria para detección de fraude.

## Dataset
Se genera un dataset sintético de 5,000 transacciones.

## Técnicas
- Feature engineering transaccional
- Manejo de clases desbalanceadas
- Random Forest
- Precision, Recall, F1-score
- ROC-AUC
- Feature importance


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    RocCurveDisplay
)

np.random.seed(42)


In [ ]:
n = 5000

amount = np.random.lognormal(mean=3.5, sigma=1.0, size=n).clip(1, 7000)
hour = np.random.randint(0, 24, n)
transactions_last_24h = np.random.poisson(3, n)
distance_from_home_km = np.random.exponential(12, n).clip(0, 400)
is_foreign_transaction = np.random.binomial(1, 0.08, n)
is_new_device = np.random.binomial(1, 0.15, n)
merchant_risk_score = np.random.beta(2, 5, n)
customer_avg_amount = np.random.lognormal(mean=3.4, sigma=0.8, size=n).clip(5, 3500)
channel = np.random.choice(["pos", "ecommerce", "wallet", "atm"], n, p=[0.45, 0.3, 0.2, 0.05])
merchant_category = np.random.choice(["retail", "travel", "gaming", "food", "cashout"], n, p=[0.45, 0.12, 0.08, 0.25, 0.10])

amount_vs_avg = amount / customer_avg_amount

fraud_score = (
    -5.0
    + 0.00055 * amount
    + 0.55 * is_foreign_transaction
    + 0.75 * is_new_device
    + 0.22 * transactions_last_24h
    + 0.006 * distance_from_home_km
    + 2.2 * merchant_risk_score
    + 0.35 * amount_vs_avg
    + np.where((hour <= 5), 0.45, 0)
    + np.where(channel == "ecommerce", 0.35, 0)
    + np.where(merchant_category == "cashout", 0.55, 0)
    + np.where(merchant_category == "gaming", 0.3, 0)
)

prob_fraud = 1 / (1 + np.exp(-fraud_score))
fraud = np.random.binomial(1, prob_fraud)

df = pd.DataFrame({
    "amount": amount.round(2),
    "hour": hour,
    "transactions_last_24h": transactions_last_24h,
    "distance_from_home_km": distance_from_home_km.round(2),
    "is_foreign_transaction": is_foreign_transaction,
    "is_new_device": is_new_device,
    "merchant_risk_score": merchant_risk_score.round(3),
    "customer_avg_amount": customer_avg_amount.round(2),
    "amount_vs_avg": amount_vs_avg.round(3),
    "channel": channel,
    "merchant_category": merchant_category,
    "fraud": fraud
})

df.head()


In [ ]:
df.shape, df["fraud"].value_counts(normalize=True).round(3)

In [ ]:
df.describe(include="all")

In [ ]:
df["amount"].hist(bins=50)
plt.title("Distribución de montos transaccionales")
plt.xlabel("amount")
plt.ylabel("frequency")
plt.show()


In [ ]:
X = df.drop(columns=["fraud"])
y = df["fraud"]

numeric_features = [
    "amount", "hour", "transactions_last_24h", "distance_from_home_km",
    "is_foreign_transaction", "is_new_device", "merchant_risk_score",
    "customer_avg_amount", "amount_vs_avg"
]
categorical_features = ["channel", "merchant_category"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=250,
        max_depth=10,
        min_samples_leaf=10,
        class_weight="balanced",
        random_state=42
    ))
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

model.fit(X_train, y_train)


In [ ]:
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print("ROC-AUC:", round(roc_auc_score(y_test, y_proba), 4))
print("\nMatriz de confusión:")
print(confusion_matrix(y_test, y_pred))
print("\nReporte de clasificación:")
print(classification_report(y_test, y_pred))


In [ ]:
RocCurveDisplay.from_predictions(y_test, y_proba)
plt.title("Curva ROC - Detección de fraude")
plt.show()


In [ ]:
feature_names = model.named_steps["preprocessor"].get_feature_names_out()
importances = model.named_steps["classifier"].feature_importances_

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values("importance", ascending=False)

importance_df.head(15)


In [ ]:
importance_df.head(15).plot(kind="barh", x="feature", y="importance", legend=False)
plt.title("Top 15 variables más importantes")
plt.gca().invert_yaxis()
plt.show()
